# Voice AI Conversational Agent — demo end-to-end (stubs)

Este notebook recorre el pipeline completo (VAD → STT → reasoner → TTS → logger) con los backends
determinísticos, así que se ejecuta sin llaves de API ni GPU. Con `ANTHROPIC_API_KEY` en el entorno el
reasoner pasa a ser `ClaudeReasoner` automáticamente.

**Todo lo que se ve aquí es salida del fallback determinista, sin LLM.**

In [1]:
import os, sys, json, asyncio
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault("LOG_LEVEL", "WARNING")
from src.config import Settings
from src.agent.loop import VoiceLoop
from src.api.schemas import TurnRequest
from src.observability import configure_logging
settings = Settings(WHISPER_BACKEND="stub", TTS_BACKEND="stub", LOGGER_BACKEND="memory", LOG_LEVEL="WARNING")
configure_logging(settings)
loop = VoiceLoop(settings)
print("reasoner:", loop.pipeline.reasoner.name, "| stt:", loop.pipeline.stt.name, "| tts:", loop.pipeline.tts.name)

/home/user/trabajo/voice-ai-agent/.venv/lib/python3.11/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


reasoner: stub | stt: stub | tts: stub


## 1. Una reserva completa en seis turnos

In [2]:
utterances = [
    "Hi, I'd like to book a table.",
    "Four people, please.",
    "Tomorrow evening.",
    "7:30 pm.",
    "Under the name John Smith.",
    "Yes, that's right.",
]
async def run(session, turns):
    out = []
    for u in turns:
        r = await loop.turn(TurnRequest(session_id=session, text=u))
        out.append(r)
        print(f"[{r.turn}] user : {u}")
        print(f"    agent: {r.response_text}")
        print(f"    action={r.action} finished={r.finished} total={r.latency.total_ms} ms slots={[(s.name, s.value) for s in r.slots]}")
        if r.finished:
            break
    return out
responses = await run("nb-1", utterances)

[1] user : Hi, I'd like to book a table.
    agent: Welcome to the restaurant, I can help you book a table. How many people will be dining?
    action=ask_clarification finished=False total=8 ms slots=[]
[2] user : Four people, please.
    agent: What date would you like to book?
    action=ask_clarification finished=False total=3 ms slots=[('party_size', 4)]
[3] user : Tomorrow evening.
    agent: What time works best for you?
    action=ask_clarification finished=False total=3 ms slots=[('party_size', 4), ('date', 'tomorrow')]
[4] user : 7:30 pm.
    agent: And under what name should I make the booking?
    action=ask_clarification finished=False total=4 ms slots=[('party_size', 4), ('date', 'tomorrow'), ('time', '19:30')]
[5] user : Under the name John Smith.
    agent: Just to confirm: a table for 4 on tomorrow at 19:30 under the name John Smith. Shall I book it?
    action=ask_clarification finished=False total=4 ms slots=[('party_size', 4), ('date', 'tomorrow'), ('time', '19:30')

## 2. Transferencia a humano y ruta de silencio (VAD)

In [3]:
import numpy as np
from src.agent.vad import encode_wav_b64
_ = await run("nb-2", ["Hello", "I'd rather talk to a real person."])
silence = encode_wav_b64(np.zeros(16_000, dtype=np.float32), 16_000)
r = await loop.turn(TurnRequest(session_id="nb-3", audio_b64=silence))
print("speech_detected:", r.speech_detected, "| reply:", r.response_text, "| backends:", r.backends)

[1] user : Hello
    agent: Welcome to the restaurant, I can help you book a table. How many people will be dining?
    action=ask_clarification finished=False total=4 ms slots=[]
[2] user : I'd rather talk to a real person.
    agent: Of course, let me transfer you to a colleague. One moment please.
    action=transfer_to_human finished=True total=5 ms slots=[]
speech_detected: False | reply: Sorry, I didn't catch that. Could you say it again? | backends: {'vad': 'energy', 'tts': 'stub'}


## 3. Transcript persistido con PII redactada y latencia por nodo

In [4]:
t = loop.transcript("nb-1")
for turn in t.turns:
    print(turn["turn"], "|", turn["transcript"], "|", turn["action"], "|", {k: round(v, 3) for k, v in turn["latency"].items()})
print("slots finales:", t.turns[-1]["slots"])

1 | Hi, I'd like to book a table. | ask_clarification | {'vad': 0.009, 'stt': 0.015, 'llm': 2.147, 'tts': 0.51}
2 | Four people, please. | ask_clarification | {'vad': 0.005, 'stt': 0.01, 'llm': 0.221, 'tts': 0.204}
3 | Tomorrow evening. | ask_clarification | {'vad': 0.004, 'stt': 0.01, 'llm': 0.068, 'tts': 0.213}
4 | 7:30 pm. | ask_clarification | {'vad': 0.004, 'stt': 0.014, 'llm': 0.077, 'tts': 0.263}
5 | Under the name NAME_d8347315. | ask_clarification | {'vad': 0.006, 'stt': 0.013, 'llm': 0.064, 'tts': 0.574}
6 | Yes, that's right. | book | {'vad': 0.004, 'stt': 0.011, 'llm': 0.089, 'tts': 0.413}
slots finales: [{'name': 'party_size', 'value': 4, 'confirmed': True}, {'name': 'date', 'value': 'tomorrow', 'confirmed': True}, {'name': 'time', 'value': '19:30', 'confirmed': True}, {'name': 'name', 'value': 'NAME_d8347315', 'confirmed': True}]


## 4. Evaluación offline sobre un subconjunto del eval set

In [5]:
from src.eval.dataset import load_conversations
from src.eval.simulate import simulate
from src.eval.judge import HeuristicJudge, summarize
convs = load_conversations()[:15]
report = await simulate(VoiceLoop(settings), convs, "stub_heuristic")
print(json.dumps(report.model_dump(exclude={"outcomes"}), indent=2))
verdicts = [HeuristicJudge().score(o, len(next(c for c in convs if c.id == o.id).turns)) for o in report.outcomes]
print("juez heurístico (proxy, no LLM):", summarize(verdicts))

{
  "backend": "stub_heuristic",
  "n": 15,
  "resolved": 15,
  "resolution_rate": 1.0,
  "booking_n": 15,
  "booking_resolution_rate": 1.0,
  "transfer_n": 0,
  "transfer_accuracy": 0.0,
  "slot_exact_match_rate": 1.0,
  "avg_turns_to_finish": 6.0,
  "latency_mean_ms": 3.24,
  "latency_p50_ms": 3,
  "latency_p95_ms": 4,
  "total_input_tokens": 0,
  "total_output_tokens": 0,
  "total_cost_usd": 0.0,
  "per_scenario": {
    "incremental": {
      "n": 15.0,
      "resolved": 15.0,
      "resolution_rate": 1.0
    }
  }
}
juez heurístico (proxy, no LLM): {'n': 15, 'pass_rate': 1.0, 'task_completion': 1.0, 'slot_correctness': 1.0, 'no_unnecessary_transfer': 1.0, 'efficiency': 1.0, 'naturalness_mean': 3.0}


## 5. Siguiente paso

Con `ANTHROPIC_API_KEY`, `python -m eval.run` añade las baselines con Claude y el LLM-as-judge a la misma
corrida y regenera `eval/RESULTS.md`. Las celdas pendientes de la tabla obligatoria (WER, MOS, costo)
quedan documentadas allí con su bloqueo concreto.